# P1 — EDA: Home Credit Default Risk

**Goal of this notebook (Sprint 1):** understand the data before modeling. Three questions:
1. How imbalanced is the target? (drives every metric choice later)
2. Where is the missingness, and is it informative?
3. Which raw features already separate defaulters from non-defaulters?

Everything here is defensible in an interview — read each output and note *why* it matters.
Fill in the **Findings** cell at the bottom with real numbers before you commit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

## Load data

Preferred: pull from PostgreSQL (proves the SQL pipeline works end to end).
Fallback: read the CSV directly if Postgres isn't up yet — swap the commented lines.

In [ ]:
# --- Option A: from PostgreSQL (preferred) ---
from sqlalchemy import create_engine
from src.config import db_url
engine = create_engine(db_url())
df = pd.read_sql("SELECT * FROM application_train", engine)

# --- Option B: fallback, straight from CSV ---
# df = pd.read_csv("data/application_train.csv")

print(df.shape)
df.head()

## 1. Target balance

`TARGET = 1` means the client had payment difficulties. Check how rare that is —
it dictates that plain accuracy is useless and that we care about AUC / PR-AUC and
recall at a chosen threshold.

In [ ]:
counts = df["TARGET"].value_counts()
rates = df["TARGET"].value_counts(normalize=True).round(4)
print(counts)
print(rates)
print(f"\nDefault rate: {rates.get(1, 0)*100:.2f}%  ->  imbalance ratio ~1:{counts[0]//counts[1]}")

ax = df["TARGET"].value_counts().plot(kind="bar")
ax.set_title("Target distribution (0 = repaid, 1 = difficulty)")
ax.set_xlabel("TARGET"); ax.set_ylabel("count")
plt.show()

## 2. Missingness

Which columns are most incomplete? In credit data, *whether* a field is missing is often
itself predictive (e.g. no external score on file). Note the worst offenders — we'll add
missingness flags in feature engineering rather than blindly impute.

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
miss_top = (miss[miss > 0].head(20) * 100).round(1)
print(miss_top)

miss_top.sort_values().plot(kind="barh", figsize=(7,6))
plt.title("Top 20 columns by % missing"); plt.xlabel("% missing"); plt.show()

## 3. Numeric overview

Sanity-check ranges and spot anomalies (e.g. `DAYS_*` are negative by design;
`DAYS_EMPLOYED` has a known sentinel value ~365243 that must be treated as missing).

In [ ]:
num = df.select_dtypes(include=[np.number])
num.describe().T[["count","mean","std","min","50%","max"]].head(30)

In [ ]:
# Known data-quality trap: DAYS_EMPLOYED sentinel
if "DAYS_EMPLOYED" in df.columns:
    print("Anomalous DAYS_EMPLOYED == 365243:", (df["DAYS_EMPLOYED"] == 365243).sum())
    print("As years, normal range should be ~0 to -50 (negative = days before application).")

## 4. What already separates the classes?

Look at default rate across a few candidate drivers. `EXT_SOURCE_1/2/3` (external credit
scores) are usually the strongest single signals — confirm it here.

In [ ]:
def default_rate_by_bin(col, bins=10):
    tmp = df[[col, "TARGET"]].dropna()
    tmp["bin"] = pd.qcut(tmp[col], q=bins, duplicates="drop")
    return tmp.groupby("bin")["TARGET"].mean()

for c in ["EXT_SOURCE_2", "EXT_SOURCE_3"]:
    if c in df.columns:
        print(f"\nDefault rate by {c} decile:")
        print((default_rate_by_bin(c)*100).round(2))

In [ ]:
# Categorical example: default rate by a few categorical fields
for c in ["CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS"]:
    if c in df.columns:
        print(f"\nDefault rate by {c} (%):")
        print((df.groupby(c)["TARGET"].mean()*100).round(2).sort_values(ascending=False))

## Findings — fill this in before committing

Write 4–6 bullets from what you actually saw. Example shape (replace with YOUR numbers):

- Default rate is **__%** (~1:__ imbalance) → accuracy is meaningless; use AUC / PR-AUC and recall @ threshold.
- Heaviest missingness: `______` at __% → will add missingness flags, not naive impute.
- `DAYS_EMPLOYED` has __ rows with the 365243 sentinel → treat as missing.
- Strongest raw separators observed: `EXT_SOURCE_3`, `______` (default rate swings from __% to __% across deciles).
- Categorical signal: `______` shows __% vs __% default rate across groups.

These bullets become the EDA section of the README and seed the feature-engineering plan.